# Coleta de tiles de satelite para deteccao de helipontos

Projeto P2 (CDIA - PUC-SP). Este notebook baixa tiles XYZ do ESRI World Imagery no zoom 19
(cerca de 0,27 metros por pixel, resolucao adequada para helipontos), gera um world file por
tile (georreferenciamento semelhante ao .j2w das imagens T_ORTO) e envia as imagens direto
para o repositorio do GitHub.

Split por bairro: treino e validacao usam Itaim/Faria Lima, Vila Olimpia e Brooklin/Berrini;
teste usa a Avenida Paulista.

Atribuicao obrigatoria ao reproduzir os tiles em figuras, slides ou relatorio:
Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community

In [ ]:
import math
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor

# url dos tiles do esri world imagery e cabecalho de uso educacional
URL = ('https://server.arcgisonline.com/ArcGIS/rest/services/'
       'World_Imagery/MapServer/tile/{z}/{y}/{x}')
HEADERS = {'User-Agent': 'PUC-SP-AM-Aula-Educacional/1.0'}

ALVO = 'heliponto'
ZOOM = 19
ZOOM_ESPERADO_HELIPONTO = 19
TAMANHO_MINIMO_VALIDO = 2521
RAIO_TERRA = 6378137.0
TAMANHO_TILE = 256
PASTA_BASE = '/content/coleta_helipontos'

# bbox no formato (lon_min, lat_min, lon_max, lat_max)
# papel define o destino no split por bairro: treino_val ou teste
REGIOES = {
    'itaim_faria_lima': {'bbox': (-46.694, -23.592, -46.672, -23.572), 'papel': 'treino_val'},
    'vila_olimpia': {'bbox': (-46.695, -23.604, -46.676, -23.590), 'papel': 'treino_val'},
    'brooklin_berrini': {'bbox': (-46.703, -23.624, -46.683, -23.606), 'papel': 'treino_val'},
    'avenida_paulista': {'bbox': (-46.660, -23.572, -46.640, -23.555), 'papel': 'teste'},
}

In [ ]:
def deg2tile(lat, lon, z):
    # converte latitude e longitude em graus para indice de tile x, y no zoom z
    n = 2.0 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def resolucao_metros_por_pixel(z, lat):
    # resolucao nominal em web mercator e resolucao no solo corrigida pela latitude
    res_mercator = (2.0 * math.pi * RAIO_TERRA) / (TAMANHO_TILE * (2.0 ** z))
    res_solo = res_mercator * math.cos(math.radians(lat))
    return res_mercator, res_solo


def verificar_zoom(z, lat):
    # confere se o zoom usado corresponde ao solicitado para helipontos
    res_mercator, res_solo = resolucao_metros_por_pixel(z, lat)
    print('zoom usado:', z)
    print('resolucao nominal mercator (m/px):', round(res_mercator, 4))
    print('resolucao no solo na latitude (m/px):', round(res_solo, 4))
    if ALVO == 'heliponto' and z != ZOOM_ESPERADO_HELIPONTO:
        print('atencao: para heliponto o zoom esperado e', ZOOM_ESPERADO_HELIPONTO)
    return res_solo

In [ ]:
def escrever_world_file(caminho_jpg, z, x, y):
    # gera um world file .jgw para georreferenciar o tile, semelhante ao .j2w das imagens t_orto
    # o sistema de coordenadas e o web mercator epsg 3857
    deslocamento_origem = 2.0 * math.pi * RAIO_TERRA / 2.0
    tamanho_tile_mercator = (2.0 * math.pi * RAIO_TERRA) / (2.0 ** z)
    pixel = tamanho_tile_mercator / TAMANHO_TILE
    canto_x = -deslocamento_origem + x * tamanho_tile_mercator
    canto_y = deslocamento_origem - y * tamanho_tile_mercator
    centro_x = canto_x + pixel / 2.0
    centro_y = canto_y - pixel / 2.0
    caminho_jgw = caminho_jpg[:-4] + '.jgw'
    with open(caminho_jgw, 'w') as arquivo:
        print(pixel, file=arquivo)
        print(0, file=arquivo)
        print(0, file=arquivo)
        print(-pixel, file=arquivo)
        print(centro_x, file=arquivo)
        print(centro_y, file=arquivo)
    return caminho_jgw

In [ ]:
def baixar_tile(z, x, y, pasta_saida):
    # baixa um unico tile com ate 3 tentativas e filtra placeholders pequenos
    nome = 'tile_z' + str(z) + '_x' + str(x) + '_y' + str(y) + '.jpg'
    destino = os.path.join(pasta_saida, nome)
    if os.path.exists(destino):
        return True
    tentativa = 0
    while tentativa < 3:
        try:
            resposta = requests.get(URL.format(z=z, x=x, y=y), headers=HEADERS, timeout=20)
            if resposta.status_code == 200 and len(resposta.content) > TAMANHO_MINIMO_VALIDO:
                with open(destino, 'wb') as arquivo:
                    arquivo.write(resposta.content)
                escrever_world_file(destino, z, x, y)
                return True
        except requests.RequestException:
            pass
        tentativa = tentativa + 1
        time.sleep(1)
    # registra a falha indicando as coordenadas e continua sem interromper
    print('falha ao baixar tile z=' + str(z) + ' x=' + str(x) + ' y=' + str(y))
    return False


def listar_tiles(bbox, z):
    # converte o bbox em uma lista de coordenadas de tile a baixar
    lon_min, lat_min, lon_max, lat_max = bbox
    x_min, y_max = deg2tile(lat_min, lon_min, z)
    x_max, y_min = deg2tile(lat_max, lon_max, z)
    if x_min > x_max:
        x_min, x_max = x_max, x_min
    if y_min > y_max:
        y_min, y_max = y_max, y_min
    coordenadas = []
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            coordenadas.append((z, x, y))
    return coordenadas


def coletar_regiao(nome, bbox, papel, z):
    # baixa todos os tiles de uma regiao para a pasta do seu papel no split
    pasta_saida = os.path.join(PASTA_BASE, papel, nome)
    os.makedirs(pasta_saida, exist_ok=True)
    lon_min, lat_min, lon_max, lat_max = bbox
    lat_centro = (lat_min + lat_max) / 2.0
    print('coletando regiao', nome, 'papel', papel, 'bbox', bbox)
    verificar_zoom(z, lat_centro)
    coordenadas = listar_tiles(bbox, z)
    sucesso = 0
    with ThreadPoolExecutor(max_workers=4) as executor:
        tarefas = []
        for z_tile, x, y in coordenadas:
            tarefas.append(executor.submit(baixar_tile, z_tile, x, y, pasta_saida))
        for tarefa in tarefas:
            if tarefa.result():
                sucesso = sucesso + 1
    print('regiao', nome, 'tiles solicitados', len(coordenadas), 'tiles salvos', sucesso)
    return len(coordenadas), sucesso

In [ ]:
# executa a coleta de todas as regioes no zoom escolhido
os.makedirs(PASTA_BASE, exist_ok=True)
total_solicitado = 0
total_salvo = 0
for nome in REGIOES:
    bbox = REGIOES[nome]['bbox']
    papel = REGIOES[nome]['papel']
    solicitado, salvo = coletar_regiao(nome, bbox, papel, ZOOM)
    total_solicitado = total_solicitado + solicitado
    total_salvo = total_salvo + salvo
print('coleta concluida. total solicitado', total_solicitado, 'total salvo', total_salvo)
print('Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community')

## Baixar as imagens para o seu PC (zip)

A celula abaixo compacta tudo em um arquivo zip e baixa para o seu computador. Depois voce
coloca essas imagens no repositorio local e sobe para o GitHub quando quiser. Sem token.

Passos:

1. Rode a celula de coleta acima e espere terminar.
2. Rode a celula abaixo. No Colab vai aparecer o download do arquivo coleta_helipontos.zip.
3. No seu PC, descompacte o zip dentro da pasta do repositorio (por exemplo em YOLOv26_Maps/dataset_bruto).
4. Suba para o GitHub de uma das formas:
   - GitHub Desktop: abra o app, ele mostra os arquivos novos, escreva uma mensagem, clique em Commit e depois Push.
   - Pelo navegador: na pagina do repositorio, use Add file > Upload files, arraste as imagens e clique em Commit changes.

In [ ]:
import shutil

# compacta a pasta coletada em um unico arquivo zip
caminho_zip = shutil.make_archive('coleta_helipontos', 'zip', PASTA_BASE)
print('arquivo zip gerado em:', caminho_zip)

# se estiver no google colab, baixa o zip direto para o seu pc
try:
    from google.colab import files
    files.download(caminho_zip)
except Exception:
    print('fora do colab: o zip ja esta salvo no caminho acima, e so abrir a pasta')